## Installing Dependencies

RAG pipeline :
documnet loading
embedding models
vector databases
llm models

In [1]:
print("Installing Dependencies\n")

!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    langchain-google-genai \
    langchain-openai \
    langchain-core \
    faiss-cpu \
    pypdf \
    sentence-transformers \
    transformers \
    torch \
    huggingface_hub \
    groq \
    langsmith \
    python-dotenv \
    tiktoken

print('\nInstallation completed')

Installing Dependencies


Installation completed


In [2]:
LLM_PROVIDER = "groq"
LLM_MODEL = "openai/gpt-oss-20b"

CORPUS_PATH = "/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus"
10
print(f"Provider: {LLM_PROVIDER}")
print(f"Model: {LLM_MODEL}")
print(f"Corpus path: {CORPUS_PATH}")

Provider: groq
Model: openai/gpt-oss-20b
Corpus path: /kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
if os.getenv("GROQ_API_KEY"):
    print("GROQ API KEY is imported!")
else:
    print("GROQ API KEY is not found!")
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

GROQ API KEY is imported!


C:\Users\Shamil\AppData\Local\Temp\ipykernel_12216\52937466.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
C:\Users\Shamil\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
CORPUS_PATH = "./zyro-dynamics-hr-corpus/"
loader = PyPDFDirectoryLoader(CORPUS_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 39 documents


In [5]:
splitter = RecursiveCharacterTextSplitter(
chunk_size = 1200,
chunk_overlap =400
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")



Created 83 chunks


## Embeddings
HuggingFaceEmbeddings


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
print("Embeddings model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4264.69it/s]


Embeddings model loaded


## Vector DB + Retrieval
FAISS Vector DB

In [16]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, model) #chunk of the data + embedding model
#retrieval
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

print("Retriever ready")

Retriever ready


In [17]:
docs = retriever.invoke(
    "How many days of Earned Leave can be carried forward?"
)

for i, doc in enumerate(docs):
    print("\n--- CHUNK", i + 1, "---")
    print(doc.page_content[:1000])


--- CHUNK 1 ---
practitioner, to be submitted within 3 working days of returning to work.
 Leave cannot be availed during the notice period unless specifically approved by the reporting manager and
HR.
EARNED LEAVE (EL)
Eligibility and Accrual
Earned Leave is accrued based on the length of continuous service. Employees become eligible for 15 days of
Earned Leave upon completion of one year of continuous service, provided they have worked for a minimum of
240 days in that year. Thereafter, Earned Leave accrues at the rate of 1.25 days per month. Employees in their
probation period accrue EL at 0.5 days per month, which becomes available for use only after probation
confirmation.
Application and Approval
Planned Earned Leave must be applied and approved at least 7 days in advance through the ZyroHR portal. All
leave requests are subject to manager approval and business requirements. Retroactive EL applications will not
be approved except in genuine emergencies, at the discretion of the

## LLM Initialization
LLM Model - groq


In [19]:
from langchain_groq import ChatGroq 
LLM_MODEL = ChatGroq(
model = 'openai/gpt-oss-120b',
temperature = 0.7,
max_tokens = 500
)
print("LLM model: groq initialized")

LLM model: groq initialized


## RAG CHAIN

In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

RAG_PROMPT = ChatPromptTemplate.from_template(
"""
You are an HR assistant. Answer employee questions using only the information provided in the HR policy context.

If the context contains the information needed to answer the question, give a clear and natural answer based on the context.

Do not use information outside the provided context and do not make up an answer.

If the question is not related to HR policies, politely explain that you can only help with HR policy-related questions.

If the question is related to HR but the required information is not available in the context, politely say that the information is not available in the provided HR policies.

Context: {context}

Question: {question}

"""
)

# all the info required for LLM

def format_docs(docs):
    return ".\n\n".join(d.page_content for d in docs)

@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    chain = RAG_PROMPT | LLM_MODEL | StrOutputParser()

    answer = chain.invoke({
        "context": context,
        "question": question
    })

    return {
        "answer": answer,
        "sources": docs
    }

## Guardials
(to ensure no context is used beyond corpus)

In [27]:
GUARDIAL_PROMPT = ChatPromptTemplate.from_template("""
You are helping employees of Zyro Dynamics with questions about their
company policies.

First, check whether the question is related to an internal HR policy such
as leave, salary, benefits, travel, performance, work from home, company
conduct, IT and data security, POSH, ESOP, or resignation.

If the question is related to the company's HR policies, return IN_SCOPE.
If it is about something unrelated to the company's HR policies, such as
general knowledge, coding, another company, or an unrelated product,
return OUT_OF_SCOPE.

Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.

Question: {question}
""")

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). "
    "I don't have information to answer that question."
)


def ask_bot(question: str):

    guardial_chain = GUARDIAL_PROMPT | LLM_MODEL | StrOutputParser()

    verdict = guardial_chain.invoke({
        "question": question
    }).strip().upper()

    if "OUT_OF_SCOPE" in verdict:
        return {
            "answer": REFUSAL_MESSAGE,
            "sources": []
        }

    return rag_chain(question)


print("Guardial Initialized!")


Guardial Initialized!


In [30]:
from langchain_groq import ChatGroq

test_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print(test_llm.invoke("Say hello").content)

Hello!


In [31]:
test_questions = [
    "How many casual leaves do I get per year?",
    "What is the internet reimbursement limit?",
    "What's the capital of France?",
]

for i, q in enumerate(test_questions, 1):
    result = ask_bot(q)

    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}")

    if result["sources"]:
        print(
            f"Sources:\n"
            f"{[d.metadata.get('source') for d in result['sources']]}"
        )

    print("-" * 60)


Q1: How many casual leaves do I get per year?
A1: You are entitled to **8 days of Casual Leave (CL) each year**.  

* Casual Leave cannot be carried forward to the next financial year and it is not encashable.  
* It is credited to your leave balance on the first day of the month following the month in which you have worked at least 20 days.  
* You may not take more than 2 consecutive days of Casual Leave; any additional days must be combined with Earned Leave.
Sources:
['zyro-dynamics-hr-corpus\\06_Compensation_and_Benefits_Policy.pdf', 'zyro-dynamics-hr-corpus\\06_Compensation_and_Benefits_Policy.pdf', 'zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\02_Leave_Policy.pdf', 'zyro-dynamics-hr-corpus\\06_Compensation_and_Benefits_Policy.pdf']
------------------------------------------------------------
Q2: What is the internet reimbursement limit?
A2: The internet reimbursement is capped at **Rs 1,000 per month** (available only to Full‑Remote employees at grade

In [32]:
from dotenv import load_dotenv
load_dotenv()

True

In [33]:
import csv

answers = []

with open("test.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    rows = list(reader)

for i, row in enumerate(rows):
    q = row["question"]

    result = ask_bot(q)

    answers.append(result["answer"])

    print(f"[{i+1}/{len(rows)}] {q}")
    print(f"->{result['answer']}")
    print("-" * 60)

[1/20] How does my Earned Leave accrue every month?
->Earned Leave (EL) builds up each month according to your length of continuous service:

- **After completing one year of service** (and having worked at least 240 days in that year), you become eligible for a base entitlement of 15 days of Earned Leave.  
- **From that point onward**, EL accrues at **1.25 days per month**.

- **During your probation period**, EL accrues at a reduced rate of **0.5 days per month**. These accrued days can be used only after your probation is confirmed.

So, once you’re past probation and have completed a year of service, you add roughly one and a quarter days of Earned Leave to your balance each month.
------------------------------------------------------------
[2/20] How much Earned Leave can I carry forward to next year?
->You can carry forward a maximum of **45 days of Earned Leave** to the next financial year (the balance is calculated as of 31 March). Any Earned Leave exceeding this limit will b

In [35]:
import pandas as pd

test_df = pd.read_csv("test.csv")

print(test_df.columns)
print(test_df.head())

Index(['question_id', 'question'], dtype='str')
  question_id                                           question
0         Q01       How does my Earned Leave accrue every month?
1         Q02  How much Earned Leave can I carry forward to n...
2         Q03  How many weeks of Maternity Leave am I entitle...
3         Q04    Do I need a medical certificate for sick leave?
4         Q05  What date does my salary get credited every mo...


In [37]:
submission = []

for _, row in test_df.iterrows():
    result = ask_bot(row["question"])

    submission.append({
        "question_id": row["question_id"],
        "answer": result["answer"]
    })

submission_df = pd.DataFrame(submission)

print(submission_df)

   question_id                                             answer
0          Q01  Earned Leave builds up each month as follows:\...
1          Q02  You can carry forward a maximum of **45 days o...
2          Q03  You are entitled to **26 weeks of paid Materni...
3          Q04  Yes. A medical certificate is required only wh...
4          Q05  Your salary (and any professional fees) is pro...
5          Q06  The CTC (cost‑to‑company) range for a Grade L4...
6          Q07  Zyro Dynamics provides **Group Medical Insuran...
7          Q08  You will be placed on a Performance Improvemen...
8          Q09  The Annual Performance Review (APR) cycle runs...
9          Q10  Based on the Work‑From‑Home policy:\n\n- **Per...
10         Q11  Yes— but only within the limits set out in the...
11         Q12  The company’s password policy for all systems ...
12         Q13  **Raising a POSH complaint**\n\n1. **Submit a ...
13         Q14  The notice period you must serve depends on yo...
14        

In [38]:
submission_df.to_csv("submission.csv", index=False)

print("submission.csv created!")

submission.csv created!


In [39]:
print("Rows:", len(submission_df))
print("Columns:", submission_df.columns.tolist())

Rows: 20
Columns: ['question_id', 'answer']


In [40]:
print(submission_df.isna().sum())
print(submission_df.head())

question_id    0
answer         0
dtype: int64
  question_id                                             answer
0         Q01  Earned Leave builds up each month as follows:\...
1         Q02  You can carry forward a maximum of **45 days o...
2         Q03  You are entitled to **26 weeks of paid Materni...
3         Q04  Yes. A medical certificate is required only wh...
4         Q05  Your salary (and any professional fees) is pro...
